# 7.21 未來答案該對到哪一格？


沿用「〈B〉／雨／停／了／〈E〉」，第二條分支讀到停時，要學的是〈E〉。讀到了時，文件卻已經沒有隔兩格的答案。若為了湊一格，拿下一篇文章的開頭當答案，就把兩份無關材料接成了同一句。

因此每條分支都要有自己的目標位置與有效範圍。這個例子的規則是：**可以預測結束標記本身，但不能越過它**；沒有目標的位置用[忽略標記−100](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)，不算直接代價。角色邊界、PAD和拼接文件也需另按資料規則處理，不能對整批資料直接循環搬移答案。

下面只計算兩份目標的交叉熵，不建立或訓練完整模型。詞表次序固定為雨、停、了、〈E〉，編號分別是0、1、2、3。人工分數全設為0，因此每個候選的softmax機率都是四分之一；我們先用這個容易手算的情況核對哪些格真的計分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
import torch.nn.functional as F

vocab = ["雨", "停", "了", "〈E〉"]
next_labels = torch.tensor([0, 1, 2, 3])
later_labels = torch.tensor([1, 2, 3, -100])
next_scores = torch.zeros(4, 4)
later_scores = torch.zeros(4, 4)

next_loss = F.cross_entropy(next_scores, next_labels, ignore_index=-100)
later_loss = F.cross_entropy(later_scores, later_labels, ignore_index=-100)
weight = 0.5
total = next_loss + weight * later_loss
print("各分支有效格", (next_labels != -100).sum().item(), (later_labels != -100).sum().item())
print("各分支平均代價", round(next_loss.item(), 4), round(later_loss.item(), 4))
print("加入輔助目標後", round(total.item(), 4))

每張分數表有四列已讀位置、四欄詞表候選。第一張的四個目標依序是雨、停、了、〈E〉；第二張是停、了、〈E〉和不計分。`cross_entropy`在每條分支內，只對有效列取平均，所以第一行輸出4和3。各列正確候選的代價都是`−log(1/4)`，平均也都是約1.3863；加上半份第二條代價，總數約2.0794。

這裡選擇「下一項平均代價＋0.5×較遠項平均代價」，用來說清楚權重和分母。0.5是本例指定，不是所有MTP方法的預設；也沒有把較少有效格誤當成第二條分支比較準。真正訓練時，用總代價的梯度更新共享核心與各分支，兩份目標都會影響學習；這段程式只做固定分數的數值核對。

如果兩篇文章被裝進同一列，還要保存它們的邊界。對較遠目標而言，不只終點需有效，從當前位置走到終點的路也不能跨越文件結束。對話微調則需明確決定哪些助理答案位置受監督；原本的下一項遮罩不能不加思考就複製給每個距離。

練習只把`weight`改成0。有效格仍是4與3，兩份分支代價仍約1.3863，總代價則只剩下一項的1.3863。若真的用這個總數更新，第二份代價不再直接提供梯度；算出第二份數字，和用它教模型，仍是兩件事。

<details>
<summary>選讀：目標移位與論文配方</summary>

若目前位置是t，下一項對應t+1，隔兩格對應t+2。這是[7.4的首答案對齊](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)向不同距離延伸，不是改變因果注意力。Gloeckle等人的[第2節](https://arxiv.org/pdf/2404.19737v1)加總不同距離的交叉熵；DeepSeek-V3的[第2.2節式24–25](https://arxiv.org/pdf/2412.19437v2)則定義額外深度的代價與權重。比較配方時應讀其完整約定，不把本節有效格平均與0.5搬過去當成相同公式。

</details>
